# Validação de Derivativos Lineares B3 (Futuros, Swaps e NDF)

Este caderno documenta a validação do módulo `bz_pricing.instruments.derivatives` e do núcleo Cython `derivatives_core` contra os ajustes oficiais da B3 (`SPRD` / `BVBG.187.01`), além da verificação analítica dos contratos `DI1`, `DDI`, `DAP`, `DOL`/`WDO`, `IND`/`WIN`, `SwapDIxPRE`, `SwapDIxIPCA`, `NDFContract` e da rotina em lote `price_di1_portfolio`.

In [1]:
import datetime, os, time
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
import pandas as pd
from bz_pricing.data.loaders import B3DataLoader
from bz_pricing.curves import CurveManager
from bz_pricing.instruments import (
    DI1Future, DDIFuture, DAPFuture, DOLFuture, INDFuture,
    SwapDIxPRE, SwapDIxIPCA, NDFContract, price_di1_portfolio,
)

ref_b3 = datetime.date(2026, 11, 12)
ldr = B3DataLoader()
cm = CurveManager(allow_download=True)

## 1. Futuros de Taxas e Inflação (`DI1`, `DDI` e `DAP`) contra o Boletim B3

In [2]:
# 1. Validação direta de DI1, DDI e DAP contra o boletim oficial da B3
val_rows = []
for asst, cls in [('DI1', DI1Future), ('DDI', DDIFuture), ('DAP', DAPFuture)]:
    df_b3 = ldr.get_curve_instruments(ref_b3, asst=asst)
    valid = df_b3.dropna(subset=['AdjstdQt', 'AdjstdQtTax'])
    valid = valid[valid['AdjstdQt'] > 0].head(3)
    for _, rec in valid.iterrows():
        tk = str(rec['TckrSymb'])
        info = cls.from_b3(tk, ref_b3, loader=ldr)
        inst = info['instrument']
        pu_calc = inst.pu(ref_b3, info['adjstd_qt_tax'])
        dv01_val = inst.dv01(ref_b3, info['adjstd_qt_tax'])
        val_rows.append({
            'ativo': asst,
            'ticker': tk,
            'vencimento': inst.maturity_date,
            'taxa_b3_%': info['adjstd_qt_tax'] * 100.0,
            'pu_calc': pu_calc,
            'pu_b3': info['adjstd_qt'],
            'diff_pu': pu_calc - info['adjstd_qt'],
            'dv01_contrato': dv01_val,
        })

pd.DataFrame(val_rows)

,ativo,ticker,vencimento,taxa_b3_%,pu_calc,pu_b3,diff_pu,dv01_contrato
0,DI1,DI1Z26,2026-12-01,13.650,99392.55,99392.55,0.0,0.416433
1,DI1,DI1F27,2027-01-04,13.662,98287.06,98287.06,0.0,1.166642
2,DI1,DI1G27,2027-02-01,13.682,97289.52,97289.52,0.0,1.833768
3,DDI,DDIZ26,2026-12-01,-24.888,101331.02,101331.02,0.0,0.270959
4,DDI,DDIF27,2027-01-04,-5.168,100766.68,100766.68,0.0,0.747430
5,DDI,DDIG27,2027-02-01,-1.419,100320.30,100320.30,0.0,1.132193
6,DAP,DAPX26,2026-11-16,11.759,99911.81,99911.81,0.0,0.070949
7,DAP,DAPZ26,2026-12-15,6.947,99415.37,99415.37,0.0,0.811493
8,DAP,DAPF27,2027-01-15,5.430,99101.79,99101.79,0.0,1.603840


## 2. Futuros de Dólar e Índice (`DOL`/`WDO`, `IND`/`WIN`), `Swap` em Decurso de Prazo, `DAP` e `NDF`

Além da marcação de futuros de dólar, índice, *swaps* e `NDF`, valida-se a consistência entre o `SwapDIxPRE` em decurso de prazo (`du_total != du_rem`) e o núcleo `swap_pre_di_mtm`, bem como o ajuste financeiro em reais de `DAPFuture.daily_settlement` com multiplicação pelo `vna_ntnb`.


In [3]:
# 2. Futuros de Dólar (DOL/WDO) e Índice (IND/WIN), Swaps e NDF
curva_pre = cm.get_curve('PRE', ref_b3)
dol = DOLFuture('DOLZ26', '2026-12-01')
wdo = DOLFuture('WDOZ26', '2026-12-01')
ind = INDFuture('INDZ26', '2026-12-16')
win = INDFuture('WINZ26', '2026-12-16')

f_dol = dol.fair_price(ref_b3, spot_ptax=5.75, pre_rate_or_curve=curva_pre, cupom_rate_or_curve=0.055)
f_ind = ind.fair_price(ref_b3, spot_ibov=128500.0, pre_rate_or_curve=curva_pre, div_yield=0.03)

sw_pre = SwapDIxPRE(notional=10_000_000.0, start_date='2026-01-02', maturity_date='2028-01-03', fixed_rate=0.145)
sw_ipca = SwapDIxIPCA(notional=10_000_000.0, start_date='2026-01-02', maturity_date='2028-01-03', ipca_coupon_rate=0.068)
ndf = NDFContract(notional_usd=1_000_000.0, maturity_date='2026-12-01', strike_brl=5.65)

pd.DataFrame([
    {'instrumento': 'DOLZ26 (fair)', 'valor': f_dol, 'ajuste_+10pts_1ctr': dol.daily_settlement(f_dol + 10, f_dol, 1)},
    {'instrumento': 'WDOZ26 (fair)', 'valor': f_dol, 'ajuste_+10pts_1ctr': wdo.daily_settlement(f_dol + 10, f_dol, 1)},
    {'instrumento': 'INDZ26 (fair)', 'valor': f_ind, 'ajuste_+100pts_1ctr': ind.daily_settlement(f_ind + 100, f_ind, 1)},
    {'instrumento': 'WINZ26 (fair)', 'valor': f_ind, 'ajuste_+100pts_1ctr': win.daily_settlement(f_ind + 100, f_ind, 1)},
    {'instrumento': 'Swap DI x Pré (MtM)', 'valor': sw_pre.mtm(ref_b3, curva_pre, di_accrued_factor=1.11), 'ajuste_+10pts_1ctr': sw_pre.risk_metrics(ref_b3, curva_pre, 1.11)['dv01']},
    {'instrumento': 'Swap DI x IPCA (MtM)', 'valor': sw_ipca.risk_metrics(ref_b3, 0.065, 1.03, 1.11)['mtm'], 'ajuste_+10pts_1ctr': sw_ipca.risk_metrics(ref_b3, 0.065, 1.03, 1.11)['dv01']},
    {'instrumento': 'NDF USD/BRL (MtM)', 'valor': ndf.mtm(ref_b3, 5.75, curva_pre, 0.055), 'ajuste_+10pts_1ctr': None},
])

# Consistência de SwapDIxPRE em decurso de prazo (du_total != du_rem) e ajuste financeiro de DAP
from bz_pricing.core.derivatives_core import swap_pre_di_mtm
from bz_pricing.core.calendars import get_calendar
from bz_pricing.instruments.derivatives import DAPFuture

cal_der = get_calendar('B3')
m_sw = sw_pre.risk_metrics(ref_b3, curva_pre, di_accrued_factor=1.11)
du_tot = cal_der.bizdays(sw_pre.start_date, sw_pre.maturity_date)
du_rem = cal_der.bizdays(ref_b3, sw_pre.maturity_date)
mtm_cy, pv_pre_cy, pv_di_cy, dv_cy = swap_pre_di_mtm(
    sw_pre.notional, sw_pre.fixed_rate, curva_pre.get_rate(du_rem), du_tot, 1.11, 1.0, du_rem
)
assert abs(m_sw['mtm'] - mtm_cy) < 1e-8
assert abs(m_sw['dv01'] - dv_cy) < 1e-8

# Swap recém-emitido na taxa zero a mercado possui MtM nulo
sw_par = SwapDIxPRE(notional=10_000_000.0, start_date=ref_b3, maturity_date='2028-01-03', fixed_rate=curva_pre.get_rate(du_rem))
mtm_par = sw_par.mtm(ref_b3, curva_pre, di_accrued_factor=1.0)
assert abs(mtm_par) < 1e-6

dap_c = DAPFuture('DAPF28', '2028-01-03')
aj_dap_pts = dap_c.daily_settlement(92500.0, 92400.0, contracts=10)
aj_dap_brl = dap_c.daily_settlement(92500.0, 92400.0, contracts=10, vna_ntnb=4771.575461)
print(f"Swap seasoned MtM={m_sw['mtm']:.2f} (du_tot={du_tot}, du_rem={du_rem}) | Swap at-par MtM={mtm_par:.2e}")
print(f"DAP ajuste +100 pts (10 ctr): pontos VNA={aj_dap_pts:.2f} | financeiro BRL={aj_dap_brl:.2f}")


Swap seasoned MtM=149053.48 (du_tot=500, du_rem=285) | Swap at-par MtM=0.00e+00
DAP ajuste +100 pts (10 ctr): pontos VNA=0.25 | financeiro BRL=1192.89


## 3. Referências Normativas e Bibliográficas

- **B3 S.A. – Brasil, Bolsa, Balcão.** *Manual de Apreçamento dos Derivativos Listados (Futuros de DI1, DDI, DAP, DOL, WDO, IND e WIN)*. São Paulo: B3. Disponível em: <https://www.b3.com.br/pt_br/market-data-e-indices/servicos-de-dados/market-data/consultas/mercado-de-derivativos/>.
- **B3 S.A. – Brasil, Bolsa, Balcão.** *Metodologia de Apreçamento de Contratos de Swap (DI x Pré e DI x IPCA) e Termo de Moedas (NDF)*. São Paulo: B3. Disponível em: <https://www.b3.com.br/>.
- **B3 S.A. – Brasil, Bolsa, Balcão.** *Boletim Diário de Informações (BDI) e Taxas Referenciais (PRE, DOC e DIC)*. São Paulo: B3. Disponível em: <https://www.b3.com.br/>.